# Model Training & Comparison
Train Random Forest and Decision Tree classifiers, compare their performance, and select the best model.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, f1_score)
import joblib

In [2]:
# load the extracted features
FEATURE_CSV = "ultrasonic_features.csv"
df = pd.read_csv(FEATURE_CSV)

X = df.drop(columns=["label"])
y = df["label"]

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Model 1: Random Forest

In [3]:
rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

## Model 2: Decision Tree

In [4]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

## Comparison Results

In [5]:
print("=" * 60)
print("MODEL COMPARISON: Random Forest vs Decision Tree")
print("=" * 60)

for name, pred in [("Random Forest", rf_pred), ("Decision Tree", dt_pred)]:
    acc = accuracy_score(y_test, pred)
    f1  = f1_score(y_test, pred, average="weighted")
    print(f"\n--- {name} ---")
    print(f"Accuracy : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"F1 Score : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, pred))

MODEL COMPARISON: Random Forest vs Decision Tree

--- Random Forest ---
Accuracy : 0.8511  (85.1%)
F1 Score : 0.8509

Classification Report:
              precision    recall  f1-score   support

      active       0.84      0.88      0.86        24
   sedentary       0.86      0.83      0.84        23

    accuracy                           0.85        47
   macro avg       0.85      0.85      0.85        47
weighted avg       0.85      0.85      0.85        47

Confusion Matrix:
[[21  3]
 [ 4 19]]

--- Decision Tree ---
Accuracy : 0.8723  (87.2%)
F1 Score : 0.8723

Classification Report:
              precision    recall  f1-score   support

      active       0.88      0.88      0.88        24
   sedentary       0.87      0.87      0.87        23

    accuracy                           0.87        47
   macro avg       0.87      0.87      0.87        47
weighted avg       0.87      0.87      0.87        47

Confusion Matrix:
[[21  3]
 [ 3 20]]


## 5-Fold Cross-Validation

In [6]:
rf_cv = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
dt_cv = cross_val_score(dt, X, y, cv=5, scoring="accuracy")
print(f"Random Forest : {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})")
print(f"Decision Tree : {dt_cv.mean():.4f} (+/- {dt_cv.std():.4f})")

Random Forest : 0.8757 (+/- 0.0777)
Decision Tree : 0.8326 (+/- 0.1001)


## Summary Table

In [7]:
print(f"{'Model':<20} {'Accuracy':>10} {'CV Mean':>10} {'CV Std':>10}")
print("-" * 52)
print(f"{'Random Forest':<20} {accuracy_score(y_test, rf_pred):>10.4f} "
      f"{rf_cv.mean():>10.4f} {rf_cv.std():>10.4f}")
print(f"{'Decision Tree':<20} {accuracy_score(y_test, dt_pred):>10.4f} "
      f"{dt_cv.mean():>10.4f} {dt_cv.std():>10.4f}")

Model                  Accuracy    CV Mean     CV Std
----------------------------------------------------
Random Forest            0.8511     0.8757     0.0777
Decision Tree            0.8723     0.8326     0.1001


## Save Best Model

In [8]:
joblib.dump(rf, "rf_activity_model.pkl")
print("Best model (Random Forest) saved as rf_activity_model.pkl")

Best model (Random Forest) saved as rf_activity_model.pkl
